# SageMaker XGBoost Training - AI Bet Prediction Engine

This notebook trains an XGBoost model to predict betting outcomes based on:
- Game statistics and odds
- User emotional state and behavioral patterns
- Historical betting performance
- Team/player performance metrics

## Prerequisites
1. Upload `training_data_sagemaker.csv` to S3
2. Have AWS credentials configured
3. Run this in SageMaker Studio or Notebook instance

In [ ]:
# Install required packages
!pip install sagemaker boto3 pandas scikit-learn --upgrade

In [ ]:
import sagemaker
import boto3
import pandas as pd
from sagemaker import get_execution_role
from sagemaker.inputs import TrainingInput
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

# Get SageMaker session and role
sagemaker_session = sagemaker.Session()
role = get_execution_role()
region = boto3.Session().region_name

print(f"SageMaker role: {role}")
print(f"Region: {region}")

## 1. Upload Training Data to S3

In [ ]:
# Configuration
bucket = sagemaker_session.default_bucket()  # Or specify your own bucket
prefix = 'serenity-ai-bet-predictor'

# Local file path
local_training_file = 'training_data_sagemaker.csv'

# Upload to S3
s3_training_data = sagemaker_session.upload_data(
    path=local_training_file,
    bucket=bucket,
    key_prefix=f'{prefix}/data'
)

print(f"Training data uploaded to: {s3_training_data}")

## 2. Inspect the Data

In [ ]:
# Load and inspect data
df = pd.read_csv(local_training_file)

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

print("\nLabel distribution:")
print(df['label'].value_counts())

print("\nFeature columns:")
print(df.columns.tolist())

print("\nBasic statistics:")
print(df.describe())

## 3. Get XGBoost Container Image

In [ ]:
from sagemaker.image_uris import retrieve

# Get the XGBoost container image
container = retrieve('xgboost', region, version='1.7-1')
print(f"XGBoost container: {container}")

## 4. Configure and Train the Model

In [ ]:
from sagemaker.estimator import Estimator

# Output path for model artifacts
output_path = f's3://{bucket}/{prefix}/output'

# Create XGBoost estimator
xgb_estimator = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',  # Start with this, scale up if needed
    output_path=output_path,
    sagemaker_session=sagemaker_session
)

# Set hyperparameters for binary classification
xgb_estimator.set_hyperparameters(
    objective='binary:logistic',  # Binary classification (win=1, loss=0)
    num_round=100,                # Number of boosting rounds
    max_depth=5,                  # Maximum tree depth
    eta=0.2,                      # Learning rate
    gamma=4,                      # Minimum loss reduction for split
    min_child_weight=6,           # Minimum sum of weights in child
    subsample=0.8,                # Subsample ratio of training instances
    verbosity=1,                  # Print messages
    eval_metric='auc'             # Evaluation metric (AUC for binary classification)
)

print("Estimator configured successfully")

In [ ]:
# Define training input
train_input = TrainingInput(
    s3_data=s3_training_data,
    content_type='text/csv'
)

# Start training
print("Starting training job...")
xgb_estimator.fit({'train': train_input})
print("Training completed!")

## 5. Deploy the Model as an Endpoint

In [ ]:
# Deploy the model
xgb_predictor = xgb_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',  # Use smaller instance for inference
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer()
)

print(f"Model deployed to endpoint: {xgb_predictor.endpoint_name}")

## 6. Test the Endpoint

In [ ]:
# Test with a sample prediction
# Format: All features except label (50 features total)
# Example: NBA Moneyline bet, low risk, good matchup
test_data = [
    # sport_nba, sport_nfl, sport_mlb, sport_nhl
    1, 0, 0, 0,
    # bet_type_moneyline, bet_type_spread, bet_type_over_under, bet_type_parlay
    1, 0, 0, 0,
    # bet_amount, budget, budget_percent
    50, 1000, 5,
    # favorite_win_pct, underdog_win_pct, has_key_injuries, bad_weather, is_parlay
    0.70, 0.40, 0, 0, 0,
    # emotional_risk_low, emotional_risk_moderate, emotional_risk_high
    1, 0, 0,
    # emotional_state_neutral, emotional_state_positive, emotional_state_negative, emotional_state_dangerous
    1, 0, 0, 0,
    # chasing_behavior, high_frequency_betting, recent_loss, recent_win
    0, 0, 0, 0,
    # favorite_ppg, underdog_ppg, favorite_fg_pct, underdog_fg_pct
    25.5, 19.2, 0.485, 0.445,
    # favorite_def_rating, underdog_def_rating, home_advantage
    105.2, 112.8, 1,
    # odds_american, implied_probability, confidence_score
    -180, 64.3, 80,
    # safety_risk_low, safety_risk_moderate, safety_risk_high
    1, 0, 0,
    # user_experience_novice, user_experience_intermediate, user_experience_expert
    0, 1, 0,
    # emotional_intensity
    2,
    # bias_overconfidence, bias_recency, bias_loss_aversion, bias_confirmation
    0, 0, 0, 0,
    # last_5_win_rate
    0.75,
    # user_age_18_25, user_age_26_35, user_age_36_45, user_age_46_plus
    0, 1, 0, 0
]

# Make prediction
result = xgb_predictor.predict(test_data)
probability = result[0]

print(f"Win Probability: {probability:.2%}")
print(f"Prediction: {'WIN' if probability > 0.5 else 'LOSS'}")
print(f"Confidence: {abs(probability - 0.5) * 200:.1f}%")

## 7. Save Endpoint Name for Backend Integration

In [ ]:
# Save endpoint name to use in your Next.js backend
endpoint_name = xgb_predictor.endpoint_name

print("=" * 60)
print("IMPORTANT: Save this endpoint name!")
print("=" * 60)
print(f"Endpoint Name: {endpoint_name}")
print(f"Region: {region}")
print("\nAdd these to your .env.local file:")
print(f"AWS_SAGEMAKER_ENDPOINT={endpoint_name}")
print(f"AWS_REGION={region}")
print("=" * 60)

## 8. Batch Predictions (Optional)

In [ ]:
# Test multiple scenarios
test_scenarios = [
    # Scenario 1: High risk parlay bet
    [1,0,0,0, 0,0,0,1, 200,400,50, 0.55,0.50, 0,0,1, 0,1,0, 0,1,0,0, 1,1,1,0, 20.5,21.2, 0.465,0.470, 109.5,108.2, 0, 250,28.6,45, 0,1,0, 0,1,0, 6, 1,1,0,1, 0.4, 0,1,0,0],
    # Scenario 2: Safe moneyline bet
    [1,0,0,0, 1,0,0,0, 25,500,5, 0.75,0.35, 0,0,0, 1,0,0, 1,0,0,0, 0,0,0,0, 26.5,17.2, 0.495,0.438, 101.5,116.8, 1, -220,68.8,90, 1,0,0, 0,1,0, 1, 0,0,0,0, 0.8, 0,1,0,0],
]

for i, scenario in enumerate(test_scenarios, 1):
    result = xgb_predictor.predict(scenario)
    prob = result[0]
    print(f"\nScenario {i}: {prob:.2%} win probability - {'WIN' if prob > 0.5 else 'LOSS'}")

## 9. Model Performance Metrics (Optional - requires validation data)

In [ ]:
# If you split your data into train/validation sets, you can evaluate here
# For now, we'll show how to get predictions on the training data

import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Load training data
df = pd.read_csv(local_training_file)
X = df.drop('label', axis=1).values
y_true = df['label'].values

# Get predictions (in batches to avoid timeout)
batch_size = 10
predictions = []

for i in range(0, len(X), batch_size):
    batch = X[i:i+batch_size]
    batch_preds = [xgb_predictor.predict(row.tolist())[0] for row in batch]
    predictions.extend(batch_preds)

y_pred_proba = np.array(predictions)
y_pred = (y_pred_proba > 0.5).astype(int)

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_proba)

print(f"Accuracy: {accuracy:.2%}")
print(f"AUC: {auc:.3f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Loss', 'Win']))

## 10. Cleanup (Run when you're done testing)

In [ ]:
# IMPORTANT: Uncomment and run this when you want to delete the endpoint
# to avoid ongoing charges

# xgb_predictor.delete_endpoint()
# print("Endpoint deleted successfully")

## Next Steps

1. **Integrate with your backend**: Use the endpoint name to call predictions from your Next.js API
2. **Collect real data**: Replace synthetic training data with actual bet outcomes
3. **Retrain regularly**: Update the model as you collect more data
4. **Monitor performance**: Track prediction accuracy and adjust hyperparameters
5. **A/B test**: Compare ML predictions against your current AI analysis

### Backend Integration Example

See `sagemaker_backend_integration.ts` for Node.js code to call this endpoint.